In [1]:
#!/usr/bin/env python
# coding: utf-8

## CELL 6 — SFCN architecture
Defines the same 3D SFCN architecture used by the training script and instantiates one model to verify the forward pass.


In [2]:
# =========================================================
# CELL 6 — SFCN ARCHITECTURE (HPC OPTIMIZED)
# =========================================================
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, max_pool=True):
        super().__init__()
        layers = [
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            # VITAL FOR HIGH ACCURACY AT SMALL BATCH SIZES
            nn.InstanceNorm3d(out_channels, affine=True),
            nn.ReLU(inplace=True),
        ]
        if max_pool:
            layers.append(nn.MaxPool3d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class SFCN(nn.Module):
    def __init__(self, output_dim=1, dropout_p=0.5):
        super().__init__()
        self.block1 = ConvBlock(1, 32)
        self.block2 = ConvBlock(32, 64)
        self.block3 = ConvBlock(64, 128)
        self.block4 = ConvBlock(128, 256)
        self.block5 = ConvBlock(256, 256)
        self.block6 = nn.Sequential(
            nn.Conv3d(256, 64, kernel_size=1, bias=False),
            nn.InstanceNorm3d(64, affine=True),
            nn.ReLU(inplace=True),
        )
        self.global_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(64, output_dim)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.block6(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)

model_check = SFCN().to(device)
n_params = sum(p.numel() for p in model_check.parameters())

print("=" * 60)
print("CELL 6: SFCN ARCHITECTURE (HPC STABILIZED)")
print("=" * 60)
print(f"Device          : {device}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Parameters      : {n_params:,}")

# Forward-pass sanity check
with torch.no_grad():
    dummy = torch.zeros((1, 1, 160, 192, 160), device=device)
    out = model_check(dummy)

print("Forward pass    : PASS")
del model_check, dummy, out
if torch.cuda.is_available():
    torch.cuda.empty_cache()

CELL 6: SFCN ARCHITECTURE (HPC STABILIZED)
Device          : cuda
GPU             : NVIDIA GeForce RTX 5080
Parameters      : 2,949,601
Forward pass    : PASS


## CELL 7 — Dataset
Defines the dataset loader and tests one real preprocessed sample.


In [3]:
# =========================================================
# CELL 7 — DATASET (RAM CACHED + ADVANCED AUGMENTATION)
# =========================================================
import numpy as np
import torch
import torchio as tio
from torch.utils.data import Dataset

# FIX: Swapped to the standard CLI tqdm to bypass the missing ipywidgets module
from tqdm import tqdm  

class BrainDataset(Dataset):
    def __init__(self, df, is_train=False, cache_in_ram=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.cache_in_ram = cache_in_ram
        self.data_cache = []

        # Advanced augmentations to maximize model generalization
        self.transform = tio.Compose([
            tio.RandomAffine(scales=(0.95, 1.05), degrees=5, p=0.5),
            tio.RandomGamma(log_gamma=(-0.3, 0.3), p=0.3), # Helps with contrast variations
            tio.RandomNoise(std=0.05, p=0.3),
        ]) if is_train else None

        # Load safely into your 128GB RAM to bypass WSL I/O crashes
        if self.cache_in_ram:
            print(f"Caching {len(self.df)} scans into RAM (Train={is_train})...")
            for filepath in tqdm(self.df["file_path"], desc="Loading NPZ"):
                with np.load(filepath) as z:
                    # float16 takes ~33GB total for the dataset, easily fitting in 128GB
                    self.data_cache.append(z["data"].astype(np.float16))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if self.cache_in_ram:
            data = self.data_cache[idx].astype(np.float32)
        else:
            with np.load(self.df.iloc[idx]["file_path"]) as z:
                data = z["data"].astype(np.float32)

        tensor = torch.from_numpy(data).unsqueeze(0)

        if self.transform:
            tensor = self.transform(tensor)

        age = torch.tensor(float(self.df.iloc[idx]["age"]), dtype=torch.float32)
        return tensor, age

print("=" * 60)
print("CELL 7: DATASET WITH RAM CACHING DEFINED")
print("=" * 60)

/mnt/d/brain age prep and meth/wsl_env_312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CELL 7: DATASET WITH RAM CACHING DEFINED


## CELL 8 — Manifest + metadata
Builds one row per preprocessed scan, joins scan metadata and CDR/MMSE data, and keeps chronological age from CDR when available.


In [4]:
# =========================================================
# CELL 8 — MANIFEST + METADATA (LINUX PATHS)
# =========================================================
import os
import re
import glob
import zipfile
import pandas as pd

# UPDATED TO LINUX PATHS FOR WSL/HPC compatibility
SCAN_META_CSV = "/mnt/d/brain age prep and meth/okarun143_8_17_2026_12_36_37.csv"
CDR_CSV = "/mnt/d/brain age prep and meth/OASIS3_UDSb4_cdr_final.csv"
PREPROC_DIR = "/mnt/d/brain age prep and meth/T1 preprocessed"

def parse_preproc_filename(path):
    fname = os.path.basename(path)
    m = re.search(r"sub-(OAS\d+)_sess?-d(\d+)(?:_run-(\d+))?", fname)
    if not m:
        raise ValueError(f"Could not parse subject/session/run from filename: {fname}")

    subject_id, day_str, run = m.group(1), m.group(2), m.group(3)
    run = run or "01"

    return {
        "subject_id": subject_id,
        "day": int(day_str),
        "run": run,
        "mri_label": f"{subject_id}_MR_d{day_str}",
        "file_path": path,
    }

def build_manifest(preproc_dir, scan_meta_csv, cdr_csv):
    files = sorted(
        glob.glob(os.path.join(preproc_dir, "**", "*_preproc.npz"), recursive=True)
    )
    if not files:
        raise FileNotFoundError(f"No preprocessed .npz files found under {preproc_dir}")

    valid_files = [f for f in files if zipfile.is_zipfile(f)]
    corrupted = [f for f in files if not zipfile.is_zipfile(f)]

    if corrupted:
        print(f"Skipping corrupted NPZ files: {len(corrupted)}")
        for f in corrupted[:10]:
            print(" ", f)

    rows = []
    parse_failures = []

    for f in valid_files:
        try:
            rows.append(parse_preproc_filename(f))
        except ValueError as e:
            parse_failures.append(str(e))

    if parse_failures:
        print(f"Filename parsing failures: {len(parse_failures)}")
        for msg in parse_failures[:10]:
            print(" ", msg)

    manifest = pd.DataFrame(rows)

    scan_meta = (
        pd.read_csv(scan_meta_csv)[["Label", "Age", "M/F"]]
        .rename(columns={"Label": "mri_label", "Age": "age_meta", "M/F": "sex"})
    )

    cdr = (
        pd.read_csv(cdr_csv)[["MRI Label", "age at visit", "CDRTOT", "CDRSUM", "MMSE"]]
        .rename(columns={"MRI Label": "mri_label", "age at visit": "age_cdr"})
    )

    # Prevent duplicate metadata columns
    scan_meta = scan_meta.drop_duplicates("mri_label")
    cdr = cdr.drop_duplicates("mri_label")

    manifest = manifest.merge(scan_meta, on="mri_label", how="left")
    manifest = manifest.merge(cdr, on="mri_label", how="left")

    manifest["age"] = pd.to_numeric(
        manifest["age_cdr"], errors="coerce"
    ).fillna(
        pd.to_numeric(manifest["age_meta"], errors="coerce")
    )

    manifest = manifest.drop(columns=["age_meta", "age_cdr"])
    manifest = manifest.dropna(subset=["age"]).reset_index(drop=True)

    return manifest

manifest = build_manifest(PREPROC_DIR, SCAN_META_CSV, CDR_CSV)

print("=" * 60)
print("CELL 8: MANIFEST + METADATA")
print("=" * 60)
print(f"Scans       : {len(manifest)}")
print(f"Subjects    : {manifest['subject_id'].nunique()}")
print(f"Age range   : {manifest['age'].min():.2f} - {manifest['age'].max():.2f}")
print(f"CDR 0.0     : {(manifest['CDRTOT'] == 0.0).sum()}")
print(f"Columns     : {list(manifest.columns)}")

display(manifest.head())

CELL 8: MANIFEST + METADATA
Scans       : 3336
Subjects    : 1098
Age range   : 42.00 - 97.47
CDR 0.0     : 2321
Columns     : ['subject_id', 'day', 'run', 'mri_label', 'file_path', 'sex', 'CDRTOT', 'CDRSUM', 'MMSE', 'age']


,subject_id,day,run,mri_label,file_path,sex,CDRTOT,CDRSUM,MMSE,age
0,OAS30001,129,01,OAS30001_MR_d0129,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,65.19
1,OAS30001,129,02,OAS30001_MR_d0129,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,65.19
2,OAS30001,757,01,OAS30001_MR_d0757,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,30.0,67.17
3,OAS30001,757,02,OAS30001_MR_d0757,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,30.0,67.17
4,OAS30001,2430,01,OAS30001_MR_d2430,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,30.0,71.17


## CELL 9 — Train / Validation / Test split
Splits at the **subject level**, so scans from the same person cannot appear in multiple splits.


In [5]:

# =========================================================
# CELL 9 — SUBJECT-LEVEL TRAIN / VAL / TEST SPLIT
# =========================================================
from sklearn.model_selection import train_test_split

def subject_level_split(manifest, test_size=0.15, val_size=0.15, random_state=42):
    subjects = manifest["subject_id"].unique()

    train_subj, test_subj = train_test_split(
        subjects, test_size=test_size, random_state=random_state
    )

    train_subj, val_subj = train_test_split(
        train_subj,
        test_size=val_size / (1 - test_size),
        random_state=random_state,
    )

    train_df = manifest[manifest["subject_id"].isin(train_subj)].reset_index(drop=True)
    val_df = manifest[manifest["subject_id"].isin(val_subj)].reset_index(drop=True)
    test_df = manifest[manifest["subject_id"].isin(test_subj)].reset_index(drop=True)

    assert not (set(train_df["subject_id"]) & set(val_df["subject_id"]))
    assert not (set(train_df["subject_id"]) & set(test_df["subject_id"]))
    assert not (set(val_df["subject_id"]) & set(test_df["subject_id"]))

    return train_df, val_df, test_df


train_df, val_df, test_df = subject_level_split(manifest)

# Training/validation are restricted to cognitively normal scans.
train_cn = train_df[train_df["CDRTOT"] == 0.0].reset_index(drop=True)
val_cn = val_df[val_df["CDRTOT"] == 0.0].reset_index(drop=True)

print("=" * 60)
print("CELL 9: TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)
print(f"Train subjects : {train_df['subject_id'].nunique()} | scans: {len(train_df)}")
print(f"Val subjects   : {val_df['subject_id'].nunique()} | scans: {len(val_df)}")
print(f"Test subjects  : {test_df['subject_id'].nunique()} | scans: {len(test_df)}")
print(f"Train CN       : {len(train_cn)}")
print(f"Val CN         : {len(val_cn)}")
print(f"Test CN        : {(test_df['CDRTOT'] == 0.0).sum()}")

if len(train_cn) == 0 or len(val_cn) == 0:
    raise RuntimeError("No CDR 0.0 scans available for training/validation.")


CELL 9: TRAIN / VALIDATION / TEST SPLIT
Train subjects : 768 | scans: 2361
Val subjects   : 165 | scans: 480
Test subjects  : 165 | scans: 495
Train CN       : 1640
Val CN         : 323
Test CN        : 358


## CELL 10 — DataLoaders
Creates loaders with `batch_size=4` because the MRI tensor is 160×192×160.


In [6]:
# =========================================================
# CELL 10 — DATALOADERS (HPC OPTIMIZED - ZERO CRASH RISK)
# =========================================================
from torch.utils.data import DataLoader

BATCH_SIZE = 4  
NUM_WORKERS = 0  # Forces WSL to use the main thread, absolutely preventing deadlocks

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": False, 
    "prefetch_factor": None,     
}

train_loader = DataLoader(
    BrainDataset(train_cn, is_train=True, cache_in_ram=True), 
    shuffle=True, 
    **loader_kwargs
)

val_loader = DataLoader(
    BrainDataset(val_cn, is_train=False, cache_in_ram=True), 
    shuffle=False, 
    **loader_kwargs
)

test_loader = DataLoader(
    BrainDataset(test_df, is_train=False, cache_in_ram=True), 
    shuffle=False, 
    **loader_kwargs
)

print("=" * 70)
print("CELL 10: DATALOADERS — HPC/LAB CONFIGURATION")
print("=" * 70)
print(f"CPU workers    : {NUM_WORKERS} (Safe Mode)")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Train batches  : {len(train_loader)}")

x0, y0 = next(iter(train_loader))
print(f"Sample input   : {tuple(x0.shape)}")
print(f"Sample age     : {float(y0[0].item()):.2f}")

del x0, y0
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Caching 1640 scans into RAM (Train=True)...


Loading NPZ: 100%|██████████| 1640/1640 [02:09<00:00, 12.66it/s]


Caching 323 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 323/323 [00:25<00:00, 12.59it/s]


Caching 495 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 495/495 [00:39<00:00, 12.55it/s]


CELL 10: DATALOADERS — HPC/LAB CONFIGURATION
CPU workers    : 0 (Safe Mode)
Batch size     : 4
Train batches  : 410
Sample input   : (4, 1, 160, 192, 160)
Sample age     : 72.56


## CELL 11 — Train SFCN
Trains an ensemble of 3 SFCN models with L1 loss, Adam, early stopping, and checkpointing.


In [7]:
# =========================================================
# CELL 11 — TRAIN SFCN ENSEMBLE (MAXIMUM QUALITY)
# =========================================================
import os
import time
import numpy as np
import torch
import torch.nn as nn

N_MODELS = 3
MAX_EPOCHS_PER_MODEL = 60 # Increased for maximum convergence
PATIENCE = 8              # Increased to prevent premature early stopping
LR = 1e-4

RESULTS_DIR = "/mnt/d/brain age prep and meth/results_notebook"
CHECKPOINT_DIR = "/mnt/d/brain age prep and meth/checkpoints_notebook"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_start = time.time()
USE_AMP = torch.cuda.is_available()
amp_dtype = torch.float16

print("=" * 70)
print("CELL 11: SFCN ENSEMBLE TRAINING")
print("=" * 70)
print(f"Device        : {device}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_abs_error = 0.0
    n = 0

    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if train else None
    context = torch.enable_grad() if train else torch.no_grad()

    with context:
        for inputs, targets in loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            try:
                if USE_AMP:
                    with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=True):
                        preds = model(inputs).view_as(targets)
                        loss = criterion(preds, targets)
                else:
                    preds = model(inputs).view_as(targets)
                    loss = criterion(preds, targets)

                if train:
                    if USE_AMP:
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        optimizer.step()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print("CUDA OOM: skipping batch.")
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                raise

            bs = inputs.size(0)
            total_loss += float(loss.detach().item()) * bs
            total_abs_error += float(torch.abs(preds.detach().float() - targets).sum().item())
            n += bs

    if n == 0:
        return float("nan"), float("nan")

    return total_loss / n, total_abs_error / n

history = {}

def train_one_model(model_idx):
    best_path = os.path.join(CHECKPOINT_DIR, f"model_{model_idx}_best.pt")

    torch.manual_seed(model_idx)
    np.random.seed(model_idx)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(model_idx)

    model = SFCN().to(device)
    criterion = nn.L1Loss()
    
    # ADVANCED: AdamW is superior to Adam for deep CNN generalization
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    
    # ADVANCED: Learning rate scheduler drops LR when validation loss stagnates
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
    )

    local_history = {"train_mae": [], "val_mae": []}
    best_val = float("inf")
    patience_counter = 0

    print("\n" + "=" * 70)
    print(f"MODEL {model_idx + 1}/{N_MODELS}")
    print("=" * 70)

    for epoch in range(MAX_EPOCHS_PER_MODEL):
        t0 = time.time()

        train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_mae = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        
        scheduler.step(val_mae)
        current_lr = optimizer.param_groups[0]['lr']

        local_history["train_mae"].append(train_mae)
        local_history["val_mae"].append(val_mae)

        print(
            f"Epoch {epoch + 1:02d}/{MAX_EPOCHS_PER_MODEL} | "
            f"Train MAE={train_mae:.2f} | Val MAE={val_mae:.2f} | "
            f"LR={current_lr:.6f} | {time.time() - t0:.1f}s"
        )

        if val_mae < best_val:
            best_val = val_mae
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping at epoch {epoch + 1} (best Val MAE={best_val:.2f})")
                break

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))

    history[model_idx] = local_history
    print(f"Model {model_idx + 1} complete | Best validation MAE={best_val:.3f} years")

    return model

models = []
for i in range(N_MODELS):
    models.append(train_one_model(i))

print("\n" + "=" * 70)
print("CELL 11: ALL MODELS TRAINED")
print("=" * 70)
for i, h in history.items():
    if h["val_mae"]:
        print(f"Model {i + 1} best Val MAE: {min(h['val_mae']):.3f} years")

CELL 11: SFCN ENSEMBLE TRAINING
Device        : cuda
GPU           : NVIDIA GeForce RTX 5080

MODEL 1/3
Epoch 01/60 | Train MAE=67.48 | Val MAE=67.42 | LR=0.000100 | 119.0s
Epoch 02/60 | Train MAE=66.21 | Val MAE=66.07 | LR=0.000100 | 119.4s
Epoch 03/60 | Train MAE=64.67 | Val MAE=64.39 | LR=0.000100 | 123.3s
Epoch 04/60 | Train MAE=62.85 | Val MAE=62.44 | LR=0.000100 | 121.2s
Epoch 05/60 | Train MAE=60.82 | Val MAE=60.28 | LR=0.000100 | 127.6s
Epoch 06/60 | Train MAE=58.51 | Val MAE=57.83 | LR=0.000100 | 130.1s
Epoch 07/60 | Train MAE=55.92 | Val MAE=55.17 | LR=0.000100 | 127.0s
Epoch 08/60 | Train MAE=53.24 | Val MAE=52.27 | LR=0.000100 | 127.8s
Epoch 09/60 | Train MAE=50.05 | Val MAE=49.15 | LR=0.000100 | 128.1s
Epoch 10/60 | Train MAE=46.89 | Val MAE=45.85 | LR=0.000100 | 126.1s
Epoch 11/60 | Train MAE=43.46 | Val MAE=42.24 | LR=0.000100 | 127.7s
Epoch 12/60 | Train MAE=39.60 | Val MAE=38.38 | LR=0.000100 | 126.0s
Epoch 13/60 | Train MAE=35.91 | Val MAE=34.37 | LR=0.000100 | 129.0s

## CELL 12 — Validation results
Evaluates each trained model and the ensemble on the validation set.


In [8]:
# =========================================================
# CELL 12 — VALIDATION RESULTS
# =========================================================
import numpy as np
import torch
from torch.utils.data import DataLoader

@torch.no_grad()
def predict_df(model, df):
    # Updated to ensure is_train=False matches BrainDataset signature
    loader = DataLoader(
        BrainDataset(df, is_train=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None
    )

    preds = []
    actual = []

    model.eval()

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        p = model(inputs).detach().cpu().numpy().reshape(-1)
        preds.extend(p.tolist())
        actual.extend(targets.numpy().reshape(-1).tolist())

    return np.asarray(actual, dtype=float), np.asarray(preds, dtype=float)

val_model_preds = []

for i, model in enumerate(models):
    y_val, p_val = predict_df(model, val_cn)
    val_model_preds.append(p_val)

    mae = float(np.mean(np.abs(p_val - y_val)))
    rmse = float(np.sqrt(np.mean((p_val - y_val) ** 2)))

    print(f"Model {i + 1}: MAE={mae:.3f} | RMSE={rmse:.3f}")

val_model_preds = np.vstack(val_model_preds)
val_ensemble_pred = val_model_preds.mean(axis=0)

val_mae = float(np.mean(np.abs(val_ensemble_pred - y_val)))
val_rmse = float(np.sqrt(np.mean((val_ensemble_pred - y_val) ** 2)))

print("-" * 60)
print(f"Ensemble validation MAE : {val_mae:.3f} years")
print(f"Ensemble validation RMSE: {val_rmse:.3f} years")

Caching 323 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 323/323 [00:25<00:00, 12.71it/s]


Model 1: MAE=4.863 | RMSE=6.000
Caching 323 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 323/323 [00:24<00:00, 13.16it/s]


Model 2: MAE=4.739 | RMSE=5.874
Caching 323 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 323/323 [00:24<00:00, 13.13it/s]


Model 3: MAE=4.824 | RMSE=5.990
------------------------------------------------------------
Ensemble validation MAE : 4.716 years
Ensemble validation RMSE: 5.827 years


## CELL 13 — Test predictions
Generates ensemble predictions for the held-out test subjects. No test labels are used to train the model or fit the bias correction.


In [9]:
# =========================================================
# CELL 13 — TEST PREDICTIONS
# =========================================================
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

@torch.no_grad()
def ensemble_predict(models, df):
    loader = DataLoader(
        BrainDataset(df, is_train=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None
    )

    all_model_preds = []

    for model in models:
        model.eval()
        preds = []

        for inputs, _ in loader:
            inputs = inputs.to(device, non_blocking=True)
            preds.extend(model(inputs).detach().cpu().numpy().reshape(-1).tolist())

        all_model_preds.append(np.asarray(preds, dtype=float))

    stacked = np.vstack(all_model_preds)

    out = df.copy()
    out["predicted_age"] = stacked.mean(axis=0)
    out["predicted_age_sd"] = stacked.std(axis=0)

    return out, stacked

test_predictions, test_model_preds = ensemble_predict(models, test_df)

test_predictions.to_csv(
    os.path.join(RESULTS_DIR, "test_predictions_raw.csv"),
    index=False
)

print("=" * 60)
print("CELL 13: TEST PREDICTIONS")
print("=" * 60)
print(f"Test scans       : {len(test_predictions)}")
print(f"Models in ensemble: {len(models)}")
print(f"Prediction SD mean: {test_predictions['predicted_age_sd'].mean():.3f}")
display(test_predictions.head())

Caching 495 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 495/495 [00:38<00:00, 12.95it/s]


CELL 13: TEST PREDICTIONS
Test scans       : 495
Models in ensemble: 3
Prediction SD mean: 0.942


,subject_id,day,run,mri_label,file_path,sex,CDRTOT,CDRSUM,MMSE,age,predicted_age,predicted_age_sd
0,OAS30011,55,01,OAS30011_MR_d0055,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,78.52,76.314054,0.860897
1,OAS30011,1671,01,OAS30011_MR_d1671,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,28.0,82.73,78.049225,0.316109
2,OAS30025,210,01,OAS30025_MR_d0210,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,26.0,64.86,61.902475,1.231312
3,OAS30025,210,02,OAS30025_MR_d0210,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,26.0,64.86,61.736754,1.293605
4,OAS30025,2298,01,OAS30025_MR_d2298,/mnt/d/brain age prep and meth/T1 preprocessed...,F,0.0,0.0,26.0,71.01,67.233462,0.798235


## CELL 14 — Brain Age + BAG
Fits the age-prediction bias correction on **training CDR 0.0 scans only**, then applies it to the held-out test predictions.


In [10]:
# =========================================================
# CELL 14 — BRAIN AGE + BAG
# =========================================================
from scipy.stats import linregress
import numpy as np
import pandas as pd

# Generate training predictions for fitting the correction.
train_predictions, _ = ensemble_predict(models, train_cn)

x = train_predictions["age"].to_numpy(dtype=float)
y = train_predictions["predicted_age"].to_numpy(dtype=float)

slope, intercept, r_value, p_value, std_err = linregress(x, y)

if abs(slope) < 1e-12:
    raise RuntimeError("Bias-correction slope is too close to zero.")

test_predictions["brain_age"] = (
    test_predictions["predicted_age"] - intercept
) / slope

test_predictions["BAG"] = (
    test_predictions["brain_age"] - test_predictions["age"]
)

test_predictions.to_csv(
    os.path.join(RESULTS_DIR, "brainage_predictions.csv"),
    index=False
)

print("=" * 60)
print("CELL 14: BRAIN AGE + BAG")
print("=" * 60)
print(f"Bias correction slope  : {float(slope):.4f}")
print(f"Bias correction intercept: {float(intercept):.4f}")
print(f"Training r             : {float(r_value):.4f}")
print(f"Training n             : {len(train_predictions)}")
print(f"Mean test BAG          : {test_predictions['BAG'].mean():.3f} years")
print(f"SD test BAG            : {test_predictions['BAG'].std(ddof=1):.3f} years")
display(test_predictions[["subject_id", "mri_label", "age", "predicted_age", "brain_age", "BAG"]].head())

Caching 1640 scans into RAM (Train=False)...


Loading NPZ: 100%|██████████| 1640/1640 [02:05<00:00, 13.03it/s]


CELL 14: BRAIN AGE + BAG
Bias correction slope  : 0.7190
Bias correction intercept: 18.6083
Training r             : 0.9166
Training n             : 1640
Mean test BAG          : 0.736 years
SD test BAG            : 6.247 years


,subject_id,mri_label,age,predicted_age,brain_age,BAG
0,OAS30011,OAS30011_MR_d0055,78.52,76.314054,80.257621,1.737621
1,OAS30011,OAS30011_MR_d1671,82.73,78.049225,82.670909,-0.059091
2,OAS30025,OAS30025_MR_d0210,64.86,61.902475,60.213892,-4.646108
3,OAS30025,OAS30025_MR_d0210,64.86,61.736754,59.983407,-4.876593
4,OAS30025,OAS30025_MR_d2298,71.01,67.233462,67.628268,-3.381732


## CELL 15 — Confusion matrix
Because SFCN is a regression model, this is a **derived age-group classification analysis**, not the model's training objective.

Age groups:
- Young: `< 60`
- Middle-aged: `60–74`
- Older: `>= 75`


In [11]:
# =========================================================
# CELL 15 — DERIVED AGE-GROUP CONFUSION MATRIX
# =========================================================
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

AGE_BINS = [-np.inf, 60, 75, np.inf]
AGE_LABELS = ["Young (<60)", "Middle (60-74)", "Older (>=75)"]

cm_df = test_predictions.copy()

cm_df["true_group"] = pd.cut(
    cm_df["age"],
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False,
)

cm_df["pred_group"] = pd.cut(
    cm_df["brain_age"],
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False,
)

valid = cm_df.dropna(subset=["true_group", "pred_group"]).copy()

y_true_group = valid["true_group"].astype(str)
y_pred_group = valid["pred_group"].astype(str)

cm = confusion_matrix(
    y_true_group,
    y_pred_group,
    labels=AGE_LABELS
)

cm_table = pd.DataFrame(
    cm,
    index=[f"Actual: {x}" for x in AGE_LABELS],
    columns=[f"Predicted: {x}" for x in AGE_LABELS],
)

accuracy = float(accuracy_score(y_true_group, y_pred_group))

# Fixed typo from original logic where 'recall_recall_fscore_support' was called
precision, recall, f1, support = precision_recall_fscore_support(
    y_true_group,
    y_pred_group,
    labels=AGE_LABELS,
    zero_division=0,
)

print("=" * 60)
print("CELL 15: DERIVED AGE-GROUP CONFUSION MATRIX")
print("=" * 60)
display(cm_table)

print(f"Derived classification accuracy : {accuracy:.3f}")
print(f"Weighted precision              : {precision_recall_fscore_support(y_true_group, y_pred_group, labels=AGE_LABELS, average='weighted', zero_division=0)[0]:.3f}")
print(f"Weighted recall                 : {precision_recall_fscore_support(y_true_group, y_pred_group, labels=AGE_LABELS, average='weighted', zero_division=0)[1]:.3f}")
print(f"Weighted F1                     : {f1.mean():.3f}  [macro F1]")

for label, p, r, f, s in zip(AGE_LABELS, precision, recall, f1, support):
    print(f"{label:16s} | Precision={p:.3f} Recall={r:.3f} F1={f:.3f} Support={s}")

CELL 15: DERIVED AGE-GROUP CONFUSION MATRIX


,Predicted: Young (<60),Predicted: Middle (60-74),Predicted: Older (>=75)
Actual: Young (<60),37,19,0
Actual: Middle (60-74),26,138,78
Actual: Older (>=75),0,19,178


Derived classification accuracy : 0.713
Weighted precision              : 0.726
Weighted recall                 : 0.713
Weighted F1                     : 0.689  [macro F1]
Young (<60)      | Precision=0.587 Recall=0.661 F1=0.622 Support=56
Middle (60-74)   | Precision=0.784 Recall=0.570 F1=0.660 Support=242
Older (>=75)     | Precision=0.695 Recall=0.904 F1=0.786 Support=197


## CELL 16 — Measurement noise
Uses same-day repeat scans with the same `mri_label` and computes the repeat-difference noise floor:
`σ_SEM = SD(differences) / √2`
and
`MDC95 = 1.96 × √2 × σ_SEM`.


In [12]:
# =========================================================
# CELL 16 — SAME-DAY REPEAT MEASUREMENT NOISE
# =========================================================
import numpy as np
import pandas as pd

def build_repeat_pairs(pred_df):
    rows = []

    for mri_label, group in pred_df.groupby("mri_label"):
        group = group.sort_values("run")

        if len(group) < 2:
            continue

        r1, r2 = group.iloc[0], group.iloc[1]

        rows.append({
            "subject_id": r1["subject_id"],
            "mri_label": mri_label,
            "BAG_run1": float(r1["BAG"]),
            "BAG_run2": float(r2["BAG"]),
            "CDRTOT": r1["CDRTOT"],
        })

    return pd.DataFrame(rows)


# For the noise analysis, use the held-out test predictions.
repeat_pairs = build_repeat_pairs(test_predictions)

print("=" * 60)
print("CELL 16: MEASUREMENT NOISE")
print("=" * 60)
print(f"Same-day repeat pairs: {len(repeat_pairs)}")

if len(repeat_pairs) >= 2:
    differences = (
        repeat_pairs["BAG_run1"].to_numpy(dtype=float)
        - repeat_pairs["BAG_run2"].to_numpy(dtype=float)
    )

    sd_difference = float(np.std(differences, ddof=1))
    sigma_sem = float(sd_difference / np.sqrt(2))
    mdc95 = float(1.96 * np.sqrt(2) * sigma_sem)

    print(f"SD of repeat differences : {sd_difference:.3f} years")
    print(f"Sigma_SEM                : {sigma_sem:.3f} years")
    print(f"MDC (95%)                : {mdc95:.3f} years")

    repeat_pairs.to_csv(
        os.path.join(RESULTS_DIR, "repeat_pairs.csv"),
        index=False
    )

else:
    sigma_sem = np.nan
    mdc95 = np.nan
    print("Not enough repeat pairs to estimate a stable noise floor.")

CELL 16: MEASUREMENT NOISE
Same-day repeat pairs: 149
SD of repeat differences : 1.260 years
Sigma_SEM                : 0.891 years
MDC (95%)                : 2.469 years


## CELL 17 — Final statistics
Prints the main regression, BAG, ensemble, derived classification, and measurement-noise results in one place.


In [13]:
# =========================================================
# CELL 17 — FINAL STATISTICS
# =========================================================
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

# Primary test performance: cognitively normal test scans only.
test_cn_pred = test_predictions[test_predictions["CDRTOT"] == 0.0].copy()

if len(test_cn_pred) > 0:
    y_true = test_cn_pred["age"].to_numpy(dtype=float)
    y_pred = test_cn_pred["brain_age"].to_numpy(dtype=float)

    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))

    if len(y_true) >= 2:
        pearson_r = float(pearsonr(y_true, y_pred)[0])
    else:
        pearson_r = np.nan
else:
    mae = rmse = r2 = pearson_r = np.nan

raw_mae = float(
    mean_absolute_error(
        test_predictions["age"].to_numpy(dtype=float),
        test_predictions["predicted_age"].to_numpy(dtype=float),
    )
)

bag_mean = float(test_predictions["BAG"].mean())
bag_sd = float(test_predictions["BAG"].std(ddof=1))
ensemble_sd_mean = float(test_predictions["predicted_age_sd"].mean())

print("\n" + "=" * 70)
print("CELL 17: FINAL BRAIN-AGE RESULTS")
print("=" * 70)

print("\nDATA")
print(f"Total scans in manifest       : {len(manifest)}")
print(f"Total subjects                : {manifest['subject_id'].nunique()}")
print(f"Train / Val / Test scans      : {len(train_df)} / {len(val_df)} / {len(test_df)}")
print(f"Test CN scans                 : {len(test_cn_pred)}")

print("\nMODEL")
print(f"Models trained                : {len(models)}/{N_MODELS}")
print(f"Batch size                    : {BATCH_SIZE}")
print(f"Learning rate                 : {LR}")
print(f"Max epochs/model              : {MAX_EPOCHS_PER_MODEL}")
print(f"Early-stopping patience       : {PATIENCE}")

print("\nPRIMARY TEST REGRESSION — CDR 0.0")
print(f"MAE                           : {mae:.3f} years")
print(f"RMSE                          : {rmse:.3f} years")
print(f"Pearson r                     : {pearson_r:.3f}")
print(f"R²                            : {r2:.3f}")

print("\nRAW ENSEMBLE TEST ERROR")
print(f"MAE before bias correction    : {raw_mae:.3f} years")
print(f"Mean ensemble prediction SD   : {ensemble_sd_mean:.3f} years")

print("\nBRAIN AGE / BAG — TEST")
print(f"Mean BAG                      : {bag_mean:.3f} years")
print(f"BAG SD                        : {bag_sd:.3f} years")

print("\nDERIVED AGE-GROUP CLASSIFICATION")
print(f"Accuracy                      : {accuracy:.3f}")
print(f"Macro F1                      : {float(np.mean(f1)):.3f}")

print("\nMEASUREMENT NOISE")
if np.isfinite(sigma_sem):
    print(f"Repeat pairs                  : {len(repeat_pairs)}")
    print(f"Sigma_SEM                     : {sigma_sem:.3f} years")
    print(f"MDC (95%)                     : {mdc95:.3f} years")
else:
    print("Noise floor                   : Not estimable (<2 repeat pairs)")

print("\nResults directory:")
print(RESULTS_DIR)

# Save a compact final summary.
summary = {
    "manifest_scans": int(len(manifest)),
    "subjects": int(manifest["subject_id"].nunique()),
    "train_scans": int(len(train_df)),
    "val_scans": int(len(val_df)),
    "test_scans": int(len(test_df)),
    "test_cn_scans": int(len(test_cn_pred)),
    "models_trained": int(len(models)),
    "mae_years": mae,
    "rmse_years": rmse,
    "pearson_r": pearson_r,
    "r2": r2,
    "raw_mae_years": raw_mae,
    "mean_bag_years": bag_mean,
    "bag_sd_years": bag_sd,
    "ensemble_prediction_sd_years": ensemble_sd_mean,
    "age_group_accuracy": accuracy,
    "age_group_macro_f1": float(np.mean(f1)),
    "repeat_pairs": int(len(repeat_pairs)),
    "sigma_sem_years": None if not np.isfinite(sigma_sem) else float(sigma_sem),
    "mdc95_years": None if not np.isfinite(mdc95) else float(mdc95),
}

import json
with open(os.path.join(RESULTS_DIR, "final_statistics.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSaved: final_statistics.json")


CELL 17: FINAL BRAIN-AGE RESULTS

DATA
Total scans in manifest       : 3336
Total subjects                : 1098
Train / Val / Test scans      : 2361 / 480 / 495
Test CN scans                 : 358

MODEL
Models trained                : 3/3
Batch size                    : 4
Learning rate                 : 0.0001
Max epochs/model              : 60
Early-stopping patience       : 8

PRIMARY TEST REGRESSION — CDR 0.0
MAE                           : 4.698 years
RMSE                          : 6.190 years
Pearson r                     : 0.803
R²                            : 0.611

RAW ENSEMBLE TEST ERROR
MAE before bias correction    : 4.721 years
Mean ensemble prediction SD   : 0.942 years

BRAIN AGE / BAG — TEST
Mean BAG                      : 0.736 years
BAG SD                        : 6.247 years

DERIVED AGE-GROUP CLASSIFICATION
Accuracy                      : 0.713
Macro F1                      : 0.689

MEASUREMENT NOISE
Repeat pairs                  : 149
Sigma_SEM                  

In [3]:
# =========================================================
# CELL 18 — LONGITUDINAL TRAJECTORIES & SURVIVAL ANALYSIS (SELF-CONTAINED)
# =========================================================
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from scipy.stats import linregress, levene
import statsmodels.formula.api as smf
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

print("=" * 70)
print("CELL 18: LONGITUDINAL TRAJECTORIES & SIGNAL ISOLATION")
print("=" * 70)

# ---------------------------------------------------------
# 1. Define Missing Helper Functions
# ---------------------------------------------------------
@torch.no_grad()
def ensemble_predict_full(models_list, df):
    loader = DataLoader(
        BrainDataset(df, is_train=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None
    )
    all_preds = []
    for m in models_list:
        m.eval()
        p = []
        for inputs, _ in loader:
            p.extend(m(inputs.to(device, non_blocking=True)).detach().cpu().numpy().reshape(-1).tolist())
        all_preds.append(np.asarray(p, dtype=float))
    
    stacked = np.vstack(all_preds)
    out = df.copy()
    out["predicted_age"] = stacked.mean(axis=0)
    out["predicted_age_sd"] = stacked.std(axis=0)
    return out

def build_repeat_pairs_df(pred_df):
    rows = []
    for mri_label, group in pred_df.groupby("mri_label"):
        group = group.sort_values("run")
        if len(group) < 2: continue
        r1, r2 = group.iloc[0], group.iloc[1]
        rows.append({
            "subject_id": r1["subject_id"],
            "mri_label": mri_label,
            "BAG_run1": float(r1["BAG"]),
            "BAG_run2": float(r2["BAG"]),
            "CDRTOT": r1["CDRTOT"],
        })
    return pd.DataFrame(rows)

def get_noise_metrics(df):
    if len(df) < 2: return np.nan, np.nan, []
    d = df["BAG_run1"] - df["BAG_run2"]
    sd = np.std(d, ddof=1)
    sem = sd / np.sqrt(2)
    return sem, 1.96 * np.sqrt(2) * sem, d

# ---------------------------------------------------------
# 2. Generate Full Cohort Predictions & Bias Correction
# ---------------------------------------------------------
print("Generating ensemble predictions for the entire cohort... (This may take a minute)")
all_pred = ensemble_predict_full(models, manifest)

# Fit bias correction on TRAINING CN scans only
train_cn_pred = all_pred[all_pred["file_path"].isin(train_cn["file_path"])]
slope, intercept, _, _, _ = linregress(train_cn_pred["age"], train_cn_pred["predicted_age"])

# Apply to all
all_pred["brain_age"] = (all_pred["predicted_age"] - intercept) / slope
all_pred["BAG"] = all_pred["brain_age"] - all_pred["age"]

# ---------------------------------------------------------
# 3. Stratify Measurement Noise
# ---------------------------------------------------------
print("\n--- 1. Stratified Noise Floor ---")
repeats_df = build_repeat_pairs_df(all_pred)
normal_repeats = repeats_df[repeats_df["CDRTOT"] == 0.0]
impaired_repeats = repeats_df[repeats_df["CDRTOT"] > 0.0]

sigma_n, mdc_n, d_n = get_noise_metrics(normal_repeats)
sigma_i, mdc_i, d_i = get_noise_metrics(impaired_repeats)

if len(d_n) > 1 and len(d_i) > 1:
    stat, p_val = levene(d_n, d_i)
    print(f"CDR 0.0   (n={len(normal_repeats)}): SEM = {sigma_n:.3f} yrs, MDC = {mdc_n:.3f} yrs")
    print(f"CDR > 0.0 (n={len(impaired_repeats)}): SEM = {sigma_i:.3f} yrs, MDC = {mdc_i:.3f} yrs")
    print(f"Levene's test for unequal variance: p = {p_val:.4f}")

sigma_noise = sigma_n if not np.isnan(sigma_n) else 1.0

# ---------------------------------------------------------
# 4. Linear Mixed-Effects Trajectory Fitting
# ---------------------------------------------------------
print("\n--- 2. Linear Mixed-Effects Trajectory Fitting ---")
all_pred = all_pred.sort_values(["subject_id", "day"])
baseline_day = all_pred.groupby("subject_id")["day"].transform("min")
all_pred["t_years"] = (all_pred["day"] - baseline_day) / 365.25

eligible = all_pred.groupby("subject_id").filter(lambda g: len(g) >= 3)
lme_model = smf.mixedlm("BAG ~ t_years", data=eligible, groups=eligible["subject_id"], re_formula="~t_years").fit()

slopes = {subj: re["t_years"] + lme_model.fe_params["t_years"] for subj, re in lme_model.random_effects.items()}
raw_slopes_df = pd.DataFrame({"subject_id": list(slopes.keys()), "S_raw": list(slopes.values())})

# ---------------------------------------------------------
# 5. Signal Isolation (Clean Speed)
# ---------------------------------------------------------
print("\n--- 3. Noise Thresholding (Signal Isolation) ---")
se_list, clean_list = [], []
for _, row in raw_slopes_df.iterrows():
    t_vals = all_pred.loc[all_pred["subject_id"] == row["subject_id"], "t_years"].values
    se_s = sigma_noise / np.sqrt(np.sum((t_vals - np.mean(t_vals)) ** 2))
    se_list.append(se_s)
    clean_list.append(row["S_raw"] if abs(row["S_raw"]) > 1.96 * se_s else 0.0)

raw_slopes_df["SE_S"] = se_list
raw_slopes_df["S_clean"] = clean_list

baseline_gaps = all_pred.sort_values("t_years").groupby("subject_id").first()["BAG"].reset_index(name="C")
clean_slopes_df = raw_slopes_df.merge(baseline_gaps, on="subject_id")

print(f"Total eligible subjects (>=3 visits): {len(clean_slopes_df)}")
print(f"Subjects exceeding noise threshold: {(clean_slopes_df['S_clean'] != 0.0).sum()}")

# ---------------------------------------------------------
# 6. Clinical Survival Analysis
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("CELL 19: CLINICAL SURVIVAL ANALYSIS (COX PH)")
print("=" * 70)

# Build Survival Cohort
cdr_df = pd.read_csv(CDR_CSV)[["MRI Label", "age at visit", "CDRTOT", "sex"]]
cdr_df = cdr_df.rename(columns={"MRI Label": "mri_label", "age at visit": "age"})
cdr_df["subject_id"] = cdr_df["mri_label"].apply(lambda x: x.split("_")[0])
cdr_df["day"] = cdr_df["mri_label"].apply(lambda x: int(x.split("_d")[1]))

rows = []
for subject_id, g in cdr_df.sort_values("day").groupby("subject_id"):
    baseline = g.iloc[:3]
    if len(baseline) < 3 or not (baseline["CDRTOT"] == 0.0).all(): continue
    post = g.iloc[3:]
    converted = post[post["CDRTOT"] > 0.0]
    
    if len(converted):
        rows.append({"subject_id": subject_id, "time_to_event": (converted["day"].iloc[0] - baseline["day"].max()) / 365.25, "event_observed": 1, "age": baseline["age"].iloc[0], "sex": baseline["sex"].iloc[0], "N": 3})
    elif len(post):
        rows.append({"subject_id": subject_id, "time_to_event": (post["day"].iloc[-1] - baseline["day"].max()) / 365.25, "event_observed": 0, "age": baseline["age"].iloc[0], "sex": baseline["sex"].iloc[0], "N": 3})

surv_df = pd.DataFrame(rows).merge(clean_slopes_df, on="subject_id").dropna()
surv_df["sex"] = surv_df["sex"].map({"M": 0, "F": 1}).fillna(0)

print(f"Survival tracking cohort size: {len(surv_df)}")
print(f"Observed clinical conversions (CDR > 0.0): {surv_df['event_observed'].sum()}")

if surv_df['event_observed'].sum() >= 5:
    cols_raw = ["time_to_event", "event_observed", "S_raw", "C", "age", "sex", "N"]
    cols_clean = ["time_to_event", "event_observed", "S_clean", "C", "age", "sex", "N"]
    
    cph_raw = CoxPHFitter().fit(surv_df[cols_raw], duration_col="time_to_event", event_col="event_observed")
    cph_clean = CoxPHFitter().fit(surv_df[cols_clean], duration_col="time_to_event", event_col="event_observed")
    
    c_raw = concordance_index(surv_df["time_to_event"], -cph_raw.predict_partial_hazard(surv_df), surv_df["event_observed"])
    c_clean = concordance_index(surv_df["time_to_event"], -cph_clean.predict_partial_hazard(surv_df), surv_df["event_observed"])
    
    print(f"\nHarrell's C-index -- raw speed model:   {c_raw:.4f}")
    print(f"Harrell's C-index -- clean speed model: {c_clean:.4f}")
else:
    print("\nINSUFFICIENT CONVERSIONS: Not enough transition events to fit Cox model.")

CELL 18: LONGITUDINAL TRAJECTORIES & SIGNAL ISOLATION
Generating ensemble predictions for the entire cohort... (This may take a minute)


NameError: name 'models' is not defined

In [ ]:
# =========================================================
# CELL 19 — CONTINUOUS CLINICAL ENDPOINT (SECTION III-G.6)
# =========================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import linregress

print("=" * 70)
print("CELL 19: CONTINUOUS CLINICAL ENDPOINT (SECTION III-G.6)")
print("=" * 70)

def extract_clinical_slopes(cdr_df, baseline_window_visits=3):
    """
    Calculates the future rate of cognitive decline (slope of CDRSUM and MMSE)
    using ONLY post-baseline visits to prevent data leakage.
    """
    rows = []
    for subject_id, g in cdr_df.sort_values("day").groupby("subject_id"):
        # We need at least 2 post-baseline visits to fit a future slope
        if len(g) <= baseline_window_visits + 1:
            continue
            
        post = g.iloc[baseline_window_visits:].copy()
        post["t_years"] = post["day"] / 365.25
        
        # Fit Future CDRSUM Slope (Positive slope = worsening dementia)
        if post["CDRSUM"].notna().sum() >= 2:
            cdr_slope, _, _, _, _ = linregress(post["t_years"], post["CDRSUM"])
        else:
            cdr_slope = np.nan
            
        # Fit Future MMSE Slope (Negative slope = worsening cognition)
        if post["MMSE"].notna().sum() >= 2:
            mmse_slope, _, _, _, _ = linregress(post["t_years"], post["MMSE"])
        else:
            mmse_slope = np.nan
            
        rows.append({
            "subject_id": subject_id,
            "future_cdr_slope": cdr_slope,
            "future_mmse_slope": mmse_slope,
            "post_baseline_visits": len(post)
        })
        
    return pd.DataFrame(rows)

print("\n--- 1. Extracting Future Cognitive Decline Slopes ---")
clinical_slopes_df = extract_clinical_slopes(cdr_df, baseline_window_visits=3)
eval_df = clean_slopes_df.merge(clinical_slopes_df, on="subject_id")
eval_df = eval_df.dropna(subset=["future_cdr_slope"])

print(f"Subjects with sufficient post-baseline clinical follow-up: {len(eval_df)}")

if len(eval_df) < 5:
    print("Insufficient subjects for continuous endpoint regression.")
else:
    print("\n--- 2. Predicting Future CDR-SUM Worsening ---")
    # Regress Clean Speed against Future CDRSUM slope
    # Controlling for Baseline BAG (C) and chronological age
    X = eval_df[["S_clean", "C", "age"]]
    X = sm.add_constant(X)
    y = eval_df["future_cdr_slope"]
    
    model_cdr_clean = sm.OLS(y, X).fit()
    print("CLEAN SPEED MODEL (S_clean):")
    print(f"Coefficient: {model_cdr_clean.params['S_clean']:.4f}")
    print(f"P-value:     {model_cdr_clean.pvalues['S_clean']:.4f}")
    print(f"R-squared:   {model_cdr_clean.rsquared:.4f}")
    
    print("\nRAW SPEED MODEL (S_raw) FOR COMPARISON:")
    X_raw = eval_df[["S_raw", "C", "age"]]
    X_raw = sm.add_constant(X_raw)
    model_cdr_raw = sm.OLS(y, X_raw).fit()
    print(f"Coefficient: {model_cdr_raw.params['S_raw']:.4f}")
    print(f"P-value:     {model_cdr_raw.pvalues['S_raw']:.4f}")
    print(f"R-squared:   {model_cdr_raw.rsquared:.4f}")
    
    # Check MMSE if available
    eval_mmse = eval_df.dropna(subset=["future_mmse_slope"])
    if len(eval_mmse) >= 5:
        print("\n--- 3. Predicting Future MMSE Decline ---")
        X_mmse = sm.add_constant(eval_mmse[["S_clean", "C", "age"]])
        model_mmse_clean = sm.OLS(eval_mmse["future_mmse_slope"], X_mmse).fit()
        print(f"Clean Speed P-value for future MMSE decline: {model_mmse_clean.pvalues['S_clean']:.4f}")

print("\nMethodology complete. Review the p-values above to conclude Section III-G.6.")